# Test the gaze-only functions

Tests **two functions**, with no video and no VRS:

- **`load_gaze_raw`** (`aria_io.py:40-46`) — reads yaw / pitch / timestamps out of
  `general_eye_gaze.csv`
- **`window_gaze_samples`** (`gaze_geometry.py:67-95`) — collects every gaze sample that
  falls between two frames

**This is the cheap notebook.** It downloads only the `mps_eye_gaze` zip, about **60 KB** —
no 165 MB video, no 2.4 GB VRS, no GPU, no torch, no projectaria-tools. Under a minute.

| Cell | What it does |
|---|---|
| 1–2 | setup, load the download-links JSON |
| 3 | pick a sequence, download + unzip **only** the gaze zip |
| 4 | look at the raw CSV *before* the function touches it |
| 5 | run `load_gaze_raw` verbatim |
| 6 | verify its outputs — shapes, ordering, ranges, NaNs |
| 7 | plot yaw and pitch over time |
| 8 | what else is in the CSV that the pipeline ignores |
| **9 / S3a–d** | **`window_gaze_samples`** — windowing, alignment, unit vectors, tiling |

**In / out of the two functions**

| | Input | Output |
|---|---|---|
| `load_gaze_raw` | `seq_dir` (a folder path) | three equal-length arrays: `ts` (µs, rebased to 0), `yaw`, `pitch` (radians) |
| `window_gaze_samples` | those arrays + two frame timestamps + a projector | a dict of ~10 entries per key: `yaw`, `pitch`, `vec3d`, `x`, `y`, `oof`, plus `n` and `n_oof` |

Section 9 uses the **pinhole** projector, since the calibration route needs the 2.4 GB VRS.
That makes its `(x, y)` values untrustworthy on a fisheye — they are there to exercise the
plumbing, not to check projection accuracy, which was settled in the Stages 0–2 notebook.

## 1 — Setup

No torch, no DINOv2, no opencv, no projectaria-tools. This function only needs pandas.

In [ ]:
import os, glob, json, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| numpy", np.__version__)

## 2 — Load the download-links JSON

Upload your `aea_download_urls.json`. Links expire after ~14 days — re-download from
[projectaria.com/datasets/aea](https://www.projectaria.com/datasets/aea/) if it fails.

In [ ]:
from google.colab import files
up = files.upload()                 # pick your aea_download_urls.json
URLS_JSON = list(up.keys())[0]

# or, if it already lives in Drive:
# from google.colab import drive; drive.mount('/content/drive')
# URLS_JSON = "/content/drive/MyDrive/aea/aea_download_urls.json"

meta = json.load(open(URLS_JSON))
seqs = list(meta["sequences"])
print(f"{len(seqs)} sequences available")
print("first 5:", seqs[:5])

## 3 — Download ONLY the gaze zip

This is the whole point of this notebook: `load_gaze_raw` never touches the video or the
VRS, so neither is downloaded.

In [ ]:
SEQ     = seqs[0]                    # >>> change to test a different sequence <<<
RAW_DIR = "/content/gaze_test/raw"

e   = meta["sequences"][SEQ]
eg  = e["mps_eye_gaze"]
print(f"sequence : {SEQ}")
print(f"file     : {eg['filename']}")
print(f"size     : {eg.get('file_size_bytes', 0)/1e6:.3f} MB")

# ---- this block is the gaze half of download_sequence (aria_io.py:17-22), verbatim ----
seq_dir = os.path.join(RAW_DIR, SEQ)
os.makedirs(os.path.join(seq_dir, "eye_gaze"), exist_ok=True)
zp = os.path.join(seq_dir, eg["filename"])
if not os.path.exists(zp):
    urllib.request.urlretrieve(eg["download_url"], zp)
with zipfile.ZipFile(zp) as z:
    print("\nzip contains:", z.namelist())
    z.extractall(os.path.join(seq_dir, "eye_gaze"))

print("\nseq_dir =", seq_dir)
for root, _, fs in os.walk(seq_dir):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f"   {os.path.getsize(p)/1e3:8.1f} KB   {os.path.relpath(p, seq_dir)}")

## 4 — The raw CSV, before the function touches it

Worth seeing what the function is selecting *from*. It reads 3 columns and discards the
rest.

In [ ]:
hits = glob.glob(os.path.join(seq_dir, "eye_gaze", "**", "general_eye_gaze.csv"),
                 recursive=True)
print(f"glob found {len(hits)} file(s):")
for h in hits:
    print("   ", h)
assert len(hits) >= 1, "general_eye_gaze.csv not found -- unzip failed"
if len(hits) > 1:
    print("   !! more than one match -- load_gaze_raw silently takes [0]")

raw = pd.read_csv(hits[0])
print(f"\nshape: {raw.shape[0]:,} rows x {raw.shape[1]} columns\n")
print("columns:")
for c in raw.columns:
    used = "  <-- USED by load_gaze_raw" if c in ("tracking_timestamp_us",
                                                  "yaw_rads_cpf", "pitch_rads_cpf") else ""
    print(f"   {c:32s} {str(raw[c].dtype):10s}{used}")

print("\nfirst 3 rows of the three columns that matter:")
print(raw[["tracking_timestamp_us", "yaw_rads_cpf", "pitch_rads_cpf"]].head(3).to_string())

## 5 — The function under test

Copied **verbatim** from `src/common/aria_io.py:40-46`.

In [ ]:
def load_gaze_raw(seq_dir):
    """RAW high-rate gaze (do NOT downsample) -> (ts_us, yaw, pitch)."""
    gcsv = glob.glob(os.path.join(seq_dir, "eye_gaze", "**", "general_eye_gaze.csv"),
                     recursive=True)[0]
    g = pd.read_csv(gcsv)
    ts = g["tracking_timestamp_us"].to_numpy(); ts = ts - ts[0]
    return ts, g["yaw_rads_cpf"].to_numpy(), g["pitch_rads_cpf"].to_numpy()


g_ts, g_yaw, g_pit = load_gaze_raw(seq_dir)

print("OUTPUT")
for nm, a in [("ts (us)", g_ts), ("yaw (rad)", g_yaw), ("pitch (rad)", g_pit)]:
    print(f"   {nm:12s} shape={a.shape}  dtype={a.dtype}")
print(f"\nfirst 5 timestamps : {g_ts[:5]}")
print(f"first 5 yaw        : {np.round(g_yaw[:5], 5)}")
print(f"first 5 pitch      : {np.round(g_pit[:5], 5)}")

## 6 — Verify the outputs

Seven checks. The **strictly increasing** one matters most: everything downstream uses
`np.searchsorted` and boolean range masks on `ts`, and both silently give wrong answers on
unsorted or duplicated timestamps.

In [ ]:
dts = np.diff(g_ts) / 1e6
pos = dts[dts > 0]

checks = [
    ("three arrays, equal length",
     len(g_ts) == len(g_yaw) == len(g_pit),
     f"{len(g_ts):,} / {len(g_yaw):,} / {len(g_pit):,}"),

    ("timestamps rebased to zero",
     g_ts[0] == 0,
     f"ts[0] = {g_ts[0]}"),

    ("timestamps STRICTLY increasing",
     np.all(np.diff(g_ts) > 0),
     f"{int((np.diff(g_ts) <= 0).sum())} non-increasing gaps"),

    ("no NaN / inf in yaw or pitch",
     np.isfinite(g_yaw).all() and np.isfinite(g_pit).all(),
     f"yaw bad={int((~np.isfinite(g_yaw)).sum())}  pitch bad={int((~np.isfinite(g_pit)).sum())}"),

    ("sampling rate is steady",
     pos.std() < 0.5 * np.median(pos),
     f"median {np.median(pos)*1000:.1f} ms, std {pos.std()*1000:.1f} ms -> {1/np.median(pos):.2f} Hz"),

    ("angles are physically plausible",
     np.abs(g_yaw).max() < 1.5 and np.abs(g_pit).max() < 1.5,
     f"|yaw|max={np.abs(g_yaw).max():.3f} rad, |pitch|max={np.abs(g_pit).max():.3f} rad"),

    ("angles look like RADIANS not degrees",
     np.abs(g_yaw).max() < 3.2,
     "values beyond ~3.2 would mean the column is in degrees"),
]

print(f"sequence: {SEQ}\n")
for name, ok, detail in checks:
    print(f"   [{'PASS' if ok else 'FAIL'}]  {name:38s} {detail}")

print(f"\nSUMMARY")
print(f"   samples   : {len(g_ts):,}")
print(f"   duration  : {g_ts[-1]/1e6:.1f} s")
print(f"   rate      : ~{1/np.median(pos):.1f} Hz")
print(f"   yaw range : [{g_yaw.min():+.3f}, {g_yaw.max():+.3f}] rad   "
      f"({np.rad2deg(g_yaw.min()):+.0f} to {np.rad2deg(g_yaw.max()):+.0f} deg)")
print(f"   pitch rng : [{g_pit.min():+.3f}, {g_pit.max():+.3f}] rad   "
      f"({np.rad2deg(g_pit.min()):+.0f} to {np.rad2deg(g_pit.max()):+.0f} deg)")
print(f"\n   at 1 FPS frames, each 1 s window holds ~{1/np.median(pos):.0f} gaze samples")
print(f"   -> ~{1/np.median(pos)-1:.0f} velocities per window in the CSV")

## 7 — Plot yaw and pitch

Flat stretches are fixations; sharp vertical jumps are saccades. Pitch skewing negative is
the wearer looking down at their hands, which is normal for egocentric activity video.

In [ ]:
t = g_ts / 1e6

fig, ax = plt.subplots(3, 1, figsize=(12, 8))
ax[0].plot(t, g_yaw, lw=.7); ax[0].set_ylabel("yaw (rad)"); ax[0].axhline(0, lw=.4, c="k")
ax[0].set_title(f"raw gaze — {SEQ}")
ax[1].plot(t, g_pit, lw=.7, color="tab:orange"); ax[1].set_ylabel("pitch (rad)")
ax[1].axhline(0, lw=.4, c="k")

# first 20 s zoomed, so individual samples are visible
sel = t < 20
ax[2].plot(t[sel], g_yaw[sel], "-o", ms=3, lw=.8, label="yaw")
ax[2].plot(t[sel], g_pit[sel], "-o", ms=3, lw=.8, label="pitch")
ax[2].set_xlabel("t (s)"); ax[2].set_ylabel("rad"); ax[2].legend()
ax[2].set_title("first 20 s — each marker is one sample")
plt.tight_layout(); plt.show()

plt.figure(figsize=(5.5, 5.5))
plt.plot(g_yaw, g_pit, lw=.3, alpha=.6)
plt.xlabel("yaw (rad)"); plt.ylabel("pitch (rad)")
plt.title("gaze angle scanpath (not yet projected onto the image)")
plt.axhline(0, lw=.4, c="k"); plt.axvline(0, lw=.4, c="k")
plt.gca().set_aspect("equal"); plt.show()

## 8 — What the pipeline throws away

`load_gaze_raw` keeps 3 columns. The MPS file ships more, and some of it is useful.

In [ ]:
used   = {"tracking_timestamp_us", "yaw_rads_cpf", "pitch_rads_cpf"}
unused = [c for c in raw.columns if c not in used]

print("columns NOT read by load_gaze_raw:\n")
for c in unused:
    s = raw[c]
    if np.issubdtype(s.dtype, np.number):
        print(f"   {c:32s} min={s.min():+10.4f}  max={s.max():+10.4f}  "
              f"nan={int(s.isna().sum())}")
    else:
        print(f"   {c:32s} (non-numeric)  n_unique={s.nunique()}")

print("\nWorth knowing:")
print("  * *_low / *_high columns are the tracker's CONFIDENCE INTERVAL on the angle.")
print("    They could size a Gaussian pooling radius around the gaze cell instead of")
print("    picking one hard patch -- currently unused.")
print("  * a depth column, if present and non-constant, would beat the hardcoded")
print("    gaze_depth_m = 1.0 used by the projection.")

---

## Reading the results

| Check | If it fails |
|---|---|
| equal length | the CSV has ragged rows — inspect it by hand |
| rebased to zero | `ts - ts[0]` did not apply; every downstream window would be offset |
| **strictly increasing** | **serious** — `np.searchsorted` and the `[t0, t1)` masks both assume sorted, unique timestamps and fail silently otherwise |
| no NaN | expected occasionally (blinks); the window functions do filter them, so a few are fine |
| steady rate | large jitter means the per-sample speed denominators vary and `inst_max` becomes unreliable |
| plausible angles | values beyond ~1.5 rad suggest the wrong column, or degrees mislabelled as radians |

If all seven pass, `load_gaze_raw` is doing its job and the problem — if there is one — lies
further down the pipeline in the projection or the windowing.

**Reference values** measured on `loc5_script4_seq6_rec1`: 2,131 samples, 213.0 s, 10.0 Hz,
yaw −33 to +34 deg, pitch −41 to +19 deg, zero NaNs.

---

# 9 — Test `window_gaze_samples`

Step 3 of the pipeline: **collect every gaze sample that falls between two frames**, and
record each one as raw angles, a 3D unit vector, and a projected image coordinate.

Still no VRS, so the **pinhole** projector is used — pure trigonometry, no calibration
file. Its `(x, y)` values are *not* trustworthy on a fisheye camera; they are here purely
to exercise the plumbing. Correctness of the real projection was settled separately in the
Stages 0–2 notebook.

Since there is no video either, the 1 FPS frame timestamps Stage 1 would produce are
**simulated** as 0 s, 1 s, 2 s, … — which is exactly what `subsample_frames` generates
(`(i / native) * 1e6`).

| Cell | What it does |
|---|---|
| **S3a** | the function verbatim, plus a pinhole projector and a unit-vector sanity check |
| **S3b** | simulate 1 FPS frames, window every pair, dump window 0 in full |
| **S3c** | nine checks |
| **S3d** | plots |

## What is actually being tested

| Check | Why it matters |
|---|---|
| all lists share length `n` | yaw[3], x[3] and vec3d[3] must describe the **same instant**; a desync silently corrupts every window |
| windows tile with no loss or overlap | `[t0, t1)` must partition the timeline — a sample counted twice inflates the motion stats |
| ~10 samples per window | confirms the 10 Hz gaze rate survives windowing |
| `vec3d` are unit vectors | `arccos` of a dot product is only an angle if both vectors have length 1 |
| `vec3d` matches its own yaw/pitch | catches any misalignment between the parallel lists |
| `oof` present and counted | the out-of-FOV flag used to be discarded; this confirms it is kept |
| `project=None` pads instead of desyncing | the alignment fix — `x`/`y` used to come back empty while `yaw` had 10 entries |

In [ ]:
# ---- verbatim from src/common/gaze_geometry.py ----
def yawpitch_to_unit_vec(yaw, pitch):
    """Gaze direction as a 3D unit vector on the sphere."""
    x = np.cos(pitch) * np.sin(yaw)
    y = np.sin(pitch)
    z = np.cos(pitch) * np.cos(yaw)
    v = np.array([x, y, z], dtype=np.float64)
    return v / (np.linalg.norm(v) + 1e-8)


def yawpitch_to_norm_xy(yaw, pitch, fov_x, fov_y, yaw_sign=1, pitch_sign=-1):
    """Pinhole (prototype-grade) projection of gaze angle to normalized image (x, y)."""
    x = 0.5 + yaw_sign * np.tan(yaw) / (2 * np.tan(fov_x / 2))
    y = 0.5 + pitch_sign * np.tan(pitch) / (2 * np.tan(fov_y / 2))
    return float(np.clip(x, 0, 1)), float(np.clip(y, 0, 1))


def window_gaze_samples(g_ts_us, g_yaw, g_pit, t0_us, t1_us, project=None):
    """All valid raw gaze samples in [t0, t1): per-sample yaw, pitch, 3D unit vector,
    projected (x, y) and the out-of-FOV flag."""
    m = (g_ts_us >= t0_us) & (g_ts_us < t1_us)
    ya, pi = g_yaw[m], g_pit[m]
    good = np.isfinite(ya) & np.isfinite(pi)
    ya, pi = ya[good], pi[good]
    yaws, pitches, vecs, xs, ys, oofs = [], [], [], [], [], []
    for y, p in zip(ya, pi):
        v = yawpitch_to_unit_vec(y, p)
        yaws.append(float(y)); pitches.append(float(p))
        vecs.append((float(v[0]), float(v[1]), float(v[2])))
        if project is not None:
            out = project(y, p)                        # calib: (x, y, oof); pinhole: (x, y)
            xs.append(float(out[0])); ys.append(float(out[1]))
            oofs.append(bool(out[2]) if len(out) > 2 else False)
        else:
            # keep every list the same length so per-sample indices stay aligned
            xs.append(float("nan")); ys.append(float("nan")); oofs.append(True)
    return dict(yaw=yaws, pitch=pitches, vec3d=vecs, x=xs, y=ys, oof=oofs,
                n=len(yaws), n_oof=int(sum(oofs)))


# Pinhole projector -- pure trigonometry, no VRS needed. Its (x, y) values are NOT
# trustworthy on a fisheye camera; they are here only to exercise the plumbing.
FOV = np.deg2rad(80.0)

def project_pinhole(yaw, pitch):
    return yawpitch_to_norm_xy(yaw, pitch, FOV, FOV, yaw_sign=1, pitch_sign=-1)


print("loaded. sanity check of the helper:")
for yy, pp, want in [(0.0, 0.0, "(0,0,1) straight ahead"),
                     (np.pi/2, 0.0, "(1,0,0) hard right"),
                     (0.0, np.pi/2, "(0,1,0) straight up")]:
    print(f"   yaw={yy:.3f} pitch={pp:.3f} -> {np.round(yawpitch_to_unit_vec(yy, pp), 4)}   {want}")

In [ ]:
# S3b -- simulate the 1 FPS frame timestamps Stage 1 would produce, then window every pair
TARGET_FPS = 1.0

n_frames = int(g_ts[-1] / 1e6 * TARGET_FPS) + 1
f_ts     = np.arange(n_frames) * (1e6 / TARGET_FPS)     # 0 s, 1 s, 2 s, ... in microseconds
print(f"simulated {n_frames} frames at {TARGET_FPS} FPS -> {n_frames-1} consecutive pairs")
print(f"frame timestamps (s): {np.round(f_ts[:6]/1e6, 1)} ...\n")

wins = [window_gaze_samples(g_ts, g_yaw, g_pit, f_ts[i-1], f_ts[i], project=project_pinhole)
        for i in range(1, len(f_ts))]

w = wins[0]
print(f"WINDOW 0  ->  [{f_ts[0]/1e6:.0f} s, {f_ts[1]/1e6:.0f} s)")
print(f"   keys      : {sorted(w.keys())}")
print(f"   n         : {w['n']}      n_oof: {w['n_oof']}")
print(f"   yaw       : {np.round(w['yaw'], 4)}")
print(f"   pitch     : {np.round(w['pitch'], 4)}")
print(f"   x         : {np.round(w['x'], 4)}")
print(f"   y         : {np.round(w['y'], 4)}")
print(f"   vec3d[0]  : {np.round(w['vec3d'][0], 5)}")
print(f"   oof       : {w['oof']}")

print(f"\nlengths of every per-sample list (must all equal n = {w['n']}):")
for k in ("yaw", "pitch", "vec3d", "x", "y", "oof"):
    print(f"   {k:6s} {len(w[k])}")

In [ ]:
# S3c -- checks
valid = np.isfinite(g_yaw) & np.isfinite(g_pit)

# every list in the dict must be the same length as n -- this is the alignment guarantee
len_ok = all(len(w["yaw"]) == len(w["pitch"]) == len(w["vec3d"]) ==
             len(w["x"]) == len(w["y"]) == len(w["oof"]) == w["n"] for w in wins)

# windows must TILE [f_ts[0], f_ts[-1]) exactly: no sample lost, none counted twice
n_expected  = int(((g_ts >= f_ts[0]) & (g_ts < f_ts[-1]) & valid).sum())
n_collected = int(sum(w["n"] for w in wins))

# vec3d must be unit length, and must agree with the yaw/pitch stored beside it
all_vecs = np.array([v for w in wins for v in w["vec3d"]])
norms    = np.linalg.norm(all_vecs, axis=1)
recomp   = np.array([yawpitch_to_unit_vec(y, p)
                     for w in wins for y, p in zip(w["yaw"], w["pitch"])])
vec_err  = np.abs(all_vecs - recomp).max() if len(all_vecs) else 0.0

# x, y must be inside the image
allx = np.concatenate([w["x"] for w in wins])
ally = np.concatenate([w["y"] for w in wins])

# project=None must pad with NaN, not desync (the alignment fix)
wn = window_gaze_samples(g_ts, g_yaw, g_pit, f_ts[0], f_ts[1], project=None)
none_ok = (len(wn["x"]) == len(wn["yaw"]) == wn["n"]) and np.isnan(wn["x"]).all()

n_per = np.array([w["n"] for w in wins])

checks = [
    ("all per-sample lists share length n", len_ok,
     "yaw / pitch / vec3d / x / y / oof all == n"),
    ("windows tile with no loss or overlap", n_collected == n_expected,
     f"collected {n_collected:,} vs expected {n_expected:,}"),
    ("~10 samples per 1 s window", 8 <= np.median(n_per) <= 12,
     f"median {np.median(n_per):.0f}, min {n_per.min()}, max {n_per.max()}"),
    ("no empty windows", (n_per > 0).all(),
     f"{int((n_per == 0).sum())} empty"),
    ("vec3d are unit vectors", np.allclose(norms, 1.0, atol=1e-6),
     f"norm range [{norms.min():.8f}, {norms.max():.8f}]"),
    ("vec3d matches its own yaw/pitch", vec_err < 1e-9,
     f"max abs diff {vec_err:.2e}"),
    ("x, y inside [0, 1]", (allx >= 0).all() and (allx <= 1).all()
                           and (ally >= 0).all() and (ally <= 1).all(),
     f"x [{allx.min():.3f}, {allx.max():.3f}]  y [{ally.min():.3f}, {ally.max():.3f}]"),
    ("oof key present and counted", all("oof" in w and "n_oof" in w for w in wins),
     f"total n_oof = {sum(w['n_oof'] for w in wins)} (pinhole never flags, so 0 is correct)"),
    ("project=None pads instead of desyncing", none_ok,
     f"len(x)={len(wn['x'])} vs n={wn['n']}, all NaN={bool(np.isnan(wn['x']).all())}"),
]

print(f"sequence: {SEQ}   |   {len(wins)} windows\n")
for name, ok, detail in checks:
    print(f"   [{'PASS' if ok else 'FAIL'}]  {name:38s} {detail}")

print(f"\nBOUNDARY SPOT-CHECK (a sample must belong to exactly one window)")
edge = f_ts[1]
print(f"   window 0 = [{f_ts[0]:.0f}, {edge:.0f})   window 1 = [{edge:.0f}, {f_ts[2]:.0f})")
on_edge = int((g_ts == edge).sum())
print(f"   samples landing exactly on the shared edge: {on_edge}")
print(f"   they belong to window 1 only, because the rule is >= t0 and < t1")

clip_hits = ((allx <= 0) | (allx >= 1) | (ally <= 0) | (ally >= 1)).mean()
print(f"\n   fraction of projected samples pinned to the border: {clip_hits*100:.2f} %")
print(f"   (high is EXPECTED here -- the pinhole model saturates on a fisheye;")
print(f"    this says nothing about the calibration projection used in production)")

In [ ]:
# S3d -- plots
n_per = np.array([w["n"] for w in wins])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].bar(np.arange(len(n_per)), n_per, width=1.0)
ax[0].axhline(np.median(n_per), color="r", lw=1, ls="--",
              label=f"median {np.median(n_per):.0f}")
ax[0].set_xlabel("window index (frame pair)"); ax[0].set_ylabel("gaze samples")
ax[0].set_title("samples per 1 s window"); ax[0].legend()

allx = np.concatenate([w["x"] for w in wins])
ally = np.concatenate([w["y"] for w in wins])
ax[1].scatter(allx, ally, s=4, alpha=.3)
ax[1].set_xlim(0, 1); ax[1].set_ylim(1, 0)
ax[1].set_xlabel("x"); ax[1].set_ylabel("y")
ax[1].set_title("all projected samples (PINHOLE -- plumbing only)")

w0 = wins[len(wins)//2]
ax[2].plot(w0["x"], w0["y"], "-o", ms=6)
ax[2].scatter([w0["x"][0]], [w0["y"][0]], s=120, c="lime", ec="k", zorder=5, label="first")
ax[2].set_xlim(0, 1); ax[2].set_ylim(1, 0); ax[2].legend()
ax[2].set_title(f"one window, {w0['n']} samples (window {len(wins)//2})")
plt.tight_layout(); plt.show()

print("Left  : should sit flat at ~10. Dips mean gaze dropouts in that second.")
print("Middle: shape reflects where the wearer looked; ignore absolute positions,")
print("        the pinhole model is wrong on a fisheye.")
print("Right : one second of eye movement -- short hops are fixation drift,")
print("        long jumps are saccades.")

---

# 10 — Test `window_gaze_rates` (the 3-channel signal)

Step 4, rewritten. `window_gaze_motion` and its `arccos` path are gone; the window now
produces three signals per step:

$$\omega_{yaw}(t) = \frac{\theta(t) - \theta(t-\Delta t)}{\Delta t}
\qquad
\omega_{pitch}(t) = \frac{\phi(t) - \phi(t-\Delta t)}{\Delta t}$$

$$\omega_{mag}(t) = \sqrt{\omega_{yaw}(t)^2 + \omega_{pitch}(t)^2}$$

All in rad/s, on the **gaze clock** (`Δt` ≈ 0.1 s), not the frame clock. Channels 1 and 2
are **signed** — positive is rightward and upward relative to the headset. Channel 3 is a
magnitude, so it is never negative.

**Shape:** 10 gaze samples per 1 s window give **9 steps × 3 channels**. Stack ~22 windows
and you have the **200 × 3** buffer the architecture slide specifies for the gating CNN —
which the previous single-scalar-per-step version could not produce.

| Cell | What it does |
|---|---|
| **S4a** | the function verbatim, window 0 printed as a 9×3 table, hand-verified against the raw angles |
| **S4b** | eight checks |
| **S4c** | **writes two CSV files** and displays them |
| **S4d** | plots of the three channels |

## The two CSVs

| File | Shape | Use |
|---|---|---|
| `gaze_rates_pipeline_format.csv` | one row per frame pair | exactly what `build_similarity_csv.py` writes — rates packed into one `;`-separated cell |
| `gaze_rates_long.csv` | one row per step | exploded, easy to read and to load for training |

Both land in `/content/` and appear in the Colab file browser (folder icon, left sidebar).

## One caveat carried over from the source

Yaw and pitch are **not orthogonal on a sphere**, so `omega_mag` is not exactly the true
angular speed — the yaw term is overstated by `1/cos(pitch)`, roughly 33% at the −41° pitch
this dataset reaches. The old `arccos` formulation was exact. This is a deliberate trade:
the 3-channel form is what the gating CNN consumes. Multiply the yaw rate by `cos(pitch)`
if a sphere-exact magnitude is ever needed.

In [ ]:
# ---- verbatim from src/common/gaze_geometry.py (replaces the old window_gaze_motion) ----
def window_gaze_rates(g_ts_us, g_yaw, g_pit, t0_us, t1_us):
    """Per-step eye angular velocity in [t0, t1), as an (n-1, 3) signal.

        omega_yaw(t)   = [theta(t) - theta(t - dt)] / dt     signed, + = rightward
        omega_pitch(t) = [phi(t)   - phi(t - dt)]   / dt     signed, + = upward
        omega_mag(t)   = sqrt(omega_yaw^2 + omega_pitch^2)   always >= 0
    """
    m = (g_ts_us >= t0_us) & (g_ts_us < t1_us)
    ts, ya, pi = g_ts_us[m], g_yaw[m], g_pit[m]
    good = np.isfinite(ya) & np.isfinite(pi)
    ts, ya, pi = ts[good], ya[good], pi[good]
    if len(ts) < 2:
        return dict(rates=np.zeros((0, 3), dtype=np.float64),
                    n_samples=int(len(ts)), n_velocities=0)
    dts = np.diff(ts) / 1e6
    dts = np.where(dts <= 0, 1e-6, dts)              # n-1 gaze-clock intervals (~0.1 s)
    w_yaw = np.diff(ya) / dts                        # signal 1
    w_pit = np.diff(pi) / dts                        # signal 2
    w_mag = np.hypot(w_yaw, w_pit)                   # signal 3
    return dict(
        rates=np.stack([w_yaw, w_pit, w_mag], axis=1),   # (n-1, 3)
        n_samples=int(len(ts)), n_velocities=int(len(dts)),
    )


rates_all = [window_gaze_rates(g_ts, g_yaw, g_pit, f_ts[i-1], f_ts[i])
             for i in range(1, len(f_ts))]

r0 = rates_all[0]
print(f"WINDOW 0  ->  [{f_ts[0]/1e6:.0f} s, {f_ts[1]/1e6:.0f} s)")
print(f"   n_samples    : {r0['n_samples']}")
print(f"   n_velocities : {r0['n_velocities']}")
print(f"   rates.shape  : {r0['rates'].shape}   <- (steps, 3 channels)\n")
print("   step |  omega_yaw   omega_pitch   omega_mag     (rad/s)")
print("   -----+---------------------------------------")
for k, r in enumerate(r0["rates"]):
    print(f"   {k:4d} | {r[0]:+10.4f}  {r[1]:+11.4f}  {r[2]:10.4f}")

print(f"\nverify against the raw angles this window was built from:")
m0 = (g_ts >= f_ts[0]) & (g_ts < f_ts[1])
print(f"   yaw   : {np.round(g_yaw[m0], 4)}")
print(f"   pitch : {np.round(g_pit[m0], 4)}")
print(f"   dt    : {np.round(np.diff(g_ts[m0])/1e6, 4)} s")
print(f"   -> step 0 omega_yaw = ({g_yaw[m0][1]:.4f} - {g_yaw[m0][0]:.4f}) / "
      f"{(g_ts[m0][1]-g_ts[m0][0])/1e6:.4f} = {r0['rates'][0,0]:+.4f}")

In [ ]:
# S4b -- checks
shapes_ok = all(r["rates"].shape == (r["n_velocities"], 3) for r in rates_all)
count_ok  = all(r["n_velocities"] == r["n_samples"] - 1 for r in rates_all)

R = np.concatenate([r["rates"] for r in rates_all], axis=0)     # (total_steps, 3)
wy, wp, wm = R[:, 0], R[:, 1], R[:, 2]

# channel 3 must be exactly hypot of channels 1 and 2
mag_err = np.abs(wm - np.hypot(wy, wp)).max()

# signs must follow the raw angle differences
sign_ok = True
for i in range(1, len(f_ts)):
    m_w = (g_ts >= f_ts[i-1]) & (g_ts < f_ts[i]) & valid
    ya_w, pi_w = g_yaw[m_w], g_pit[m_w]
    if len(ya_w) < 2:
        continue
    rr = rates_all[i-1]["rates"]
    if not (np.all(np.sign(rr[:, 0]) == np.sign(np.diff(ya_w))) and
            np.all(np.sign(rr[:, 1]) == np.sign(np.diff(pi_w)))):
        sign_ok = False
        break

checks = [
    ("rates shape is (n-1, 3)", shapes_ok,
     f"e.g. window 0 -> {rates_all[0]['rates'].shape}"),
    ("n_velocities == n_samples - 1", count_ok,
     f"window 0: {rates_all[0]['n_samples']} samples -> {rates_all[0]['n_velocities']} steps"),
    ("omega_mag == hypot(yaw, pitch)", mag_err < 1e-12,
     f"max abs error {mag_err:.2e}"),
    ("omega_mag is never negative", (wm >= 0).all(),
     f"min {wm.min():.6f}"),
    ("channels 1 and 2 are SIGNED", (wy < 0).any() and (wy > 0).any()
                                    and (wp < 0).any() and (wp > 0).any(),
     f"yaw {(wy<0).mean()*100:.0f}% negative, pitch {(wp<0).mean()*100:.0f}% negative"),
    ("signs follow the raw angle differences", sign_ok,
     "d(yaw)/dt and d(pitch)/dt keep their direction"),
    ("all finite", np.isfinite(R).all(),
     f"{int((~np.isfinite(R)).sum())} non-finite values"),
    ("speeds are physically plausible", wm.max() < 50.0,
     f"max {wm.max():.2f} rad/s = {np.rad2deg(wm.max()):.0f} deg/s"),
]

print(f"sequence: {SEQ}   |   {R.shape[0]:,} steps total\n")
for name, ok, detail in checks:
    print(f"   [{'PASS' if ok else 'FAIL'}]  {name:38s} {detail}")

print(f"\nCHANNEL STATISTICS (rad/s)")
for nm, ch in [("omega_yaw  ", wy), ("omega_pitch", wp), ("omega_mag  ", wm)]:
    print(f"   {nm}  mean {ch.mean():+8.4f}   std {ch.std():7.4f}   "
          f"min {ch.min():+8.4f}   max {ch.max():+8.4f}")

print(f"\n   median omega_mag : {np.median(wm):.4f} rad/s  = {np.rad2deg(np.median(wm)):.1f} deg/s")
print(f"   95th percentile  : {np.percentile(wm, 95):.4f} rad/s  = {np.rad2deg(np.percentile(wm, 95)):.1f} deg/s")
print(f"   -> low median with a long tail is EXPECTED: fixations dominate, saccades spike")
print(f"\n   NOTE: at 10 Hz a real saccade lasts less than one sampling interval,")
print(f"   so these peaks UNDER-report true saccadic velocity (which reaches ~500 deg/s).")

In [ ]:
# S4c -- write it out as CSV, two ways

# ---------- (a) PIPELINE FORMAT: one row per frame pair, rates packed into one cell ----------
rows = []
for i in range(1, len(f_ts)):
    w, rr = wins[i-1], rates_all[i-1]
    rows.append(dict(
        idx=i-1, sequence=SEQ,
        t_start_s=round(f_ts[i-1]/1e6, 3), t_end_s=round(f_ts[i]/1e6, 3),
        n_gaze_in_gap=w["n"], n_oof_in_gap=w["n_oof"],
        yaw_pitch_window=";".join(f"{y:.5f},{p:.5f}"
                                  for y, p in zip(w["yaw"], w["pitch"])),
        n_velocities=rr["n_velocities"],
        gaze_rates_window=";".join(f"{r[0]:.5f},{r[1]:.5f},{r[2]:.5f}"
                                   for r in rr["rates"]),
    ))
df_pipe = pd.DataFrame(rows)
PIPE_CSV = "/content/gaze_rates_pipeline_format.csv"
df_pipe.to_csv(PIPE_CSV, index=False)

print("(a) PIPELINE FORMAT -- exactly what build_similarity_csv.py writes")
print(f"    {df_pipe.shape[0]} rows x {df_pipe.shape[1]} columns  ->  {PIPE_CSV}\n")
display(df_pipe.head(3))
print("\none packed gaze_rates_window cell (row 0), split back out:")
for k, trip in enumerate(df_pipe.loc[0, "gaze_rates_window"].split(";")):
    wy, wp, wm = trip.split(",")
    print(f"   step {k}:  omega_yaw={float(wy):+8.4f}   omega_pitch={float(wp):+8.4f}   omega_mag={float(wm):7.4f}")

# ---------- (b) LONG FORMAT: one row per step -- far easier to eyeball ----------
long_rows = []
for i in range(1, len(f_ts)):
    m_w  = (g_ts >= f_ts[i-1]) & (g_ts < f_ts[i]) & valid
    ts_w = g_ts[m_w]
    for k, r in enumerate(rates_all[i-1]["rates"]):
        long_rows.append(dict(
            window=i-1, step=k,
            t_s=round(ts_w[k+1]/1e6, 4),          # time of the LATER sample in the step
            omega_yaw=round(float(r[0]), 6),
            omega_pitch=round(float(r[1]), 6),
            omega_mag=round(float(r[2]), 6),
        ))
df_long = pd.DataFrame(long_rows)
LONG_CSV = "/content/gaze_rates_long.csv"
df_long.to_csv(LONG_CSV, index=False)

print(f"\n\n(b) LONG FORMAT -- one row per step  ->  {LONG_CSV}")
print(f"    {df_long.shape[0]:,} rows ({len(rates_all)} windows x ~{rates_all[0]['n_velocities']} steps)\n")
display(df_long.head(12))

print("\nsummary statistics of the three channels:")
display(df_long[["omega_yaw", "omega_pitch", "omega_mag"]].describe().round(4))

print("\nboth files are in the Colab file browser (folder icon, left sidebar).")
print("uncomment below to download them to your laptop:")
# from google.colab import files
# files.download(PIPE_CSV)
# files.download(LONG_CSV)

In [ ]:
# S4d -- plots of the three channels
fig, ax = plt.subplots(4, 1, figsize=(13, 10))

ax[0].plot(df_long["t_s"], df_long["omega_yaw"], lw=.6)
ax[0].axhline(0, lw=.4, c="k"); ax[0].set_ylabel("omega_yaw\n(rad/s)")
ax[0].set_title(f"the 3-channel gate input — {SEQ}")

ax[1].plot(df_long["t_s"], df_long["omega_pitch"], lw=.6, color="tab:orange")
ax[1].axhline(0, lw=.4, c="k"); ax[1].set_ylabel("omega_pitch\n(rad/s)")

ax[2].plot(df_long["t_s"], df_long["omega_mag"], lw=.6, color="tab:green")
ax[2].set_ylabel("omega_mag\n(rad/s)"); ax[2].set_xlabel("t (s)")
ax[2].axhline(df_long["omega_mag"].median(), lw=.8, ls="--", c="r",
              label=f"median {df_long['omega_mag'].median():.2f}")
ax[2].legend()

# one window zoomed: 9 steps, 3 channels
wsel = df_long[df_long["window"] == len(rates_all)//2]
ax[3].plot(wsel["step"], wsel["omega_yaw"],   "-o", label="omega_yaw")
ax[3].plot(wsel["step"], wsel["omega_pitch"], "-o", label="omega_pitch")
ax[3].plot(wsel["step"], wsel["omega_mag"],   "-o", label="omega_mag", color="tab:green")
ax[3].axhline(0, lw=.4, c="k"); ax[3].legend(); ax[3].set_xlabel("step within window")
ax[3].set_ylabel("rad/s"); ax[3].set_title(f"one window = {len(wsel)} steps x 3 channels")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df_long["omega_mag"], bins=60)
ax[0].set_xlabel("omega_mag (rad/s)"); ax[0].set_ylabel("count")
ax[0].set_title("magnitude distribution — long tail = saccades")
ax[1].scatter(df_long["omega_yaw"], df_long["omega_pitch"], s=3, alpha=.25)
ax[1].axhline(0, lw=.4, c="k"); ax[1].axvline(0, lw=.4, c="k")
ax[1].set_xlabel("omega_yaw"); ax[1].set_ylabel("omega_pitch")
ax[1].set_title("direction of eye movement")
ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

print("Channels 1 and 2 are SIGNED -- they cross zero as the eyes reverse direction.")
print("Channel 3 is the magnitude, so it never goes below zero.")
print("Most values sit near zero (fixations); the spikes are saccades.")